# Lot D — EXP-D-012 : Tiny Aya (3,35 Md) contre Aya Expanse 8B

**Auteur : Membre 5 (Hadil).** Même protocole qu'EXP-D-010 (`--main-only` : prompt v1, température 0.3,
top-p 0.9, 4 bits, recherche hybride top 5, graines 13 / 42 / 2026). Plan et règle de décision fixés
avant la mesure dans `rag_llm/compare_llm_runs.py`. Système principal : `tiny-aya-global` ;
secondaire : `tiny-aya-earth`. Dev d'abord, puis test gelé **une seule fois** par modèle.

Avant de commencer :
1. *Exécution → Modifier le type d'exécution → **GPU T4*** ;
2. accepter les licences (non commerciales) de https://huggingface.co/CohereLabs/tiny-aya-global
   et https://huggingface.co/CohereLabs/tiny-aya-earth avec le même compte que le jeton.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Jetons (cases masquées) et projet (branche `lot-d-tiny-aya`)

In [ ]:
import os, subprocess, base64, sys
from pathlib import Path
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Jeton Hugging Face Read : ").strip()
token = getpass("Jeton GitHub (lecture du dépôt) : ").strip()
auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
env = dict(os.environ, GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
           GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + auth, GIT_TERMINAL_PROMPT="0")
if not Path("/content/NewsCheckTN/.git").exists():
    r = subprocess.run(["git", "clone", "-q", "-b", "lot-d-tiny-aya",
                        "https://github.com/SouissiOumaima/NewsCheckTN.git", "/content/NewsCheckTN"],
                       env=env, capture_output=True)
    if r.returncode:
        raise RuntimeError("Clone privé échoué : vérifier le jeton et le nom de la branche")
r = subprocess.run(["git", "-C", "/content/NewsCheckTN", "fetch", "-q", "origin", "lot-a1-data"],
                   env=env, capture_output=True)
del token, auth, env
os.chdir("/content/NewsCheckTN")
print("✅ commit", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip(),
      "| lot-a1-data :", "OK" if r.returncode == 0 else "échec")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt",
                "sentence-transformers", "bitsandbytes", "accelerate", "-U", "transformers"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
print("✅ dépendances installées")

## 2. Corpus de test (privé, reconstruit dans la session, jamais poussé)

In [ ]:
!python -m rag_llm.build_tunifact_eval
!python -m rag_llm.freeze_test --check

## 3. Configuration (identique à EXP-D-010, seul le modèle change)
NLI Bronze pour le gabarit de repli, comme dans EXP-D-010.

In [ ]:
os.environ.update({"LLM_PROVIDER": "hf", "RETRIEVAL_METHOD": "hybrid", "NLI_BACKEND": "bronze"})
GLOBAL, EARTH = "CohereLabs/tiny-aya-global", "CohereLabs/tiny-aya-earth"

## 4. Essai rapide (3 affirmations du dev) : le modèle se charge et répond en JSON valide

In [ ]:
os.environ["LLM_MODEL"] = GLOBAL
!python -X utf8 -m rag_llm.silver_verdict --main-only --limit 3

## 5. Dev complet — tiny-aya-global
**Envoyer ce résultat à Claude avant le test.**

In [ ]:
os.environ["LLM_MODEL"] = GLOBAL
!python -X utf8 -m rag_llm.silver_verdict --main-only --splits dev

## 6. Test gelé — tiny-aya-global (une seule fois)

In [ ]:
os.environ["LLM_MODEL"] = GLOBAL
!python -X utf8 -m rag_llm.silver_verdict --main-only --splits test

## 7. Secondaire — tiny-aya-earth : dev, puis test (une seule fois)

In [ ]:
os.environ["LLM_MODEL"] = EARTH
!python -X utf8 -m rag_llm.silver_verdict --main-only --splits dev

In [ ]:
os.environ["LLM_MODEL"] = EARTH
!python -X utf8 -m rag_llm.silver_verdict --main-only --splits test

## 8. Récupérer les résultats
Contient des extraits Tunifact : **privé**, à garder sur le PC (jamais sur GitHub). La comparaison
avec Aya 8B se fait ensuite sur le PC, avec le fichier d'EXP-D-010.

In [ ]:
!cd evaluation/results && zip -q ../../lot-d_exp-d-012_tiny-aya.zip lot-d_silver_verdict_*tiny-aya*.json lot-d_llm_cache.jsonl
from google.colab import files
files.download("lot-d_exp-d-012_tiny-aya.zip")